# Predictor Saber 11: de los datos a una página web

**Curso Machine Learning 1 · Universidad EAN (Bogotá)**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel01010101010101/Ejercicio_ejemplo_machine_learning/blob/main/notebooks/saber11_modelo.ipynb)

En este cuaderno construimos, de principio a fin, un modelo de Machine Learning que estima el **puntaje global del Saber 11** (de 0 a 500) a partir del contexto del estudiante. Al final lo exportamos a un archivo `modelo.js` para publicarlo como una página web que calcula todo en el navegador.

**Cómo usarlo.** En Colab: menú *Entorno de ejecución → Ejecutar todas*. El cuaderno descarga los datos por sí solo (tarda uno o dos minutos) y al final descarga `modelo.js`, el archivo que usa la página.

| Sección | Qué hacemos |
|---|---|
| 1 | El problema y el uso responsable |
| 2 | Descarga de los datos desde la API de datos.gov.co |
| 3 | Exploración breve |
| 4 | Limpieza y elección de variables |
| 5 | Separación en entrenamiento y prueba |
| 6 | Pipeline y entrenamiento |
| 7 | Evaluación contra una línea base |
| 8 | Interpretación de coeficientes |
| 9 | Exportación a `modelo.js` y prueba de paridad |
| 10 | Publicación (GitHub → Netlify) y re-entrenamiento |

Cada paso tiene tres partes: **qué hacemos**, **por qué** y **qué error común evitamos**.

## 1. El problema y el uso responsable

**Qué hacemos.** Planteamos una **regresión**: la variable objetivo es `punt_global`, un número entre 0 y 500. Las variables de entrada describen el contexto del estudiante: su hogar (estrato, educación de la madre y del padre, internet, computador, automóvil) y su colegio (oficial o privado, jornada, zona, bilingüismo, departamento). Todas son **categóricas**.

**Por qué este problema.** Con datos abiertos y reales podemos recorrer el ciclo completo de un proyecto de ML: obtener datos, limpiarlos, entrenar, evaluar, interpretar y desplegar. El resultado deja además una lección incómoda y valiosa: el contexto social predice una parte importante del puntaje.

### Uso responsable: lo que este modelo NO es

- **Mide desigualdad de contexto, no capacidad.** Si el modelo estima un puntaje bajo para un perfil, lo que dice es que los estudiantes con ese contexto *obtuvieron en promedio* puntajes más bajos, en buena parte por brechas de oportunidades. No dice nada del talento ni del esfuerzo de una persona.
- **Nunca debe usarse para juzgar, clasificar, admitir o rechazar a un estudiante**, ni para repartir becas, cupos o expectativas. Usarlo así convertiría una desigualdad en una profecía autocumplida.
- **Se equivoca mucho a nivel individual.** En la sección 7 veremos que dos estudiantes con las mismas respuestas pueden tener puntajes muy distintos.
- **Describe asociaciones, no causas.** Comprar un carro no sube el puntaje; el carro está asociado a otras condiciones del hogar.

### Variables que dejamos fuera a propósito

- **`estu_genero` (género del estudiante).** Es una decisión ética. En los datos puede haber diferencias promedio por género, pero reflejan factores sociales (estereotipos, expectativas, oportunidades), no capacidad. Si la incluyéramos, la página le diría a cada estudiante que su puntaje esperado sube o baja *por ser hombre o mujer*: convertiríamos un estereotipo en una predicción individual. Además, el género es un atributo protegido: decidir sobre personas con base en él es discriminación.
- **Nombres y códigos de colegio** (`cole_nombre_establecimiento`, `cole_codigo_icfes`, `cole_cod_dane_*`). Identifican instituciones concretas (el modelo terminaría "calificando" colegios con nombre propio), tienen miles de categorías con pocos estudiantes cada una (sobreajuste) y no describen el contexto de forma general.

> **Error común que evitamos:** creer que "más variables siempre es mejor". Una variable puede mejorar un poco la métrica y, a la vez, volver el modelo injusto o inútil para su propósito.

Primero cargamos las librerías. Todas vienen instaladas en Google Colab.

In [ ]:
from __future__ import annotations

import datetime as dt
import io
import json
import math
import shutil
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

EN_COLAB = "google.colab" in sys.modules
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · scikit-learn {sklearn.__version__}")
print("Entorno:", "Google Colab" if EN_COLAB else "local")

Ahora la configuración del proyecto. Todo lo que podríamos querer cambiar está aquí, en un solo lugar.

> **Error común que evitamos:** repartir "números mágicos" por todo el cuaderno. Si la semilla o el porcentaje de prueba aparecen en diez celdas, tarde o temprano alguien cambia uno y olvida los otros.

In [ ]:
VERSION = "1.0.0"
DATASET_ID = "kgxf-xxbe"
API = f"https://www.datos.gov.co/resource/{DATASET_ID}"
URL_CONJUNTO = f"https://www.datos.gov.co/d/{DATASET_ID}"

OBJETIVO = "punt_global"
SIN_INFO = "Sin información"
OTRA = "Otra (pocos casos)"

# Variables candidatas del enunciado (todas categóricas, todas llegan como texto).
CANDIDATAS = [
    "fami_estratovivienda",
    "fami_educacionmadre",
    "fami_educacionpadre",
    "fami_tieneinternet",
    "fami_tienecomputador",
    "fami_tieneautomovil",
    "cole_naturaleza",
    "cole_jornada",
    "cole_area_ubicacion",
    "cole_bilingue",
    "cole_depto_ubicacion",
]

# Columnas extra que se descargan solo para filtrar o describir.
COLUMNAS_EXTRA = ["periodo", "estu_estadoinvestigacion"]

SEMILLA = 42
PROPORCION_PRUEBA = 0.2
MIN_CASOS = 200  # una categoría con menos casos en train se agrupa en OTRA
ALPHA = 1.0

PERIODO = None  # None = el periodo más reciente disponible; también puede ser, p. ej., "20224"

# Carpetas: si el cuaderno se corre dentro del repositorio, se usan las del repositorio.
RAIZ = Path.cwd().parent if (Path.cwd() / "saber11_modelo.ipynb").exists() else Path.cwd()
CARPETA_DATOS = RAIZ / "datos"   # datos crudos (no se suben a GitHub)
CARPETA_WEB = RAIZ / "web"       # aquí queda modelo.js para la página


def fmt(numero, decimales=0):
    """Formato colombiano: 12.345,6"""
    texto = f"{numero:,.{decimales}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


# Estilo de las gráficas: una paleta sobria y legible también para daltónicos.
AZUL, ROJO, GRIS, TINTA, TINTA_2 = "#2a78d6", "#e34948", "#b9b8b0", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (8.5, 4), "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2, "ytick.color": TINTA_2, "ytick.left": False,
})

## 2. Descarga de los datos desde la API

**Qué hacemos.** Descargamos los resultados de Saber 11 del conjunto **«Resultados únicos Saber 11»** del ICFES, publicado en [datos.gov.co](https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe) con el identificador `kgxf-xxbe`. El portal usa la API **Socrata**, que acepta consultas parecidas a SQL en la misma URL:

| Parámetro | Para qué sirve | Ejemplo |
|---|---|---|
| `$select` | elegir columnas o calcular | `periodo, count(*)` |
| `$where` | filtrar filas | `periodo = '20224'` |
| `$group` | agrupar | `periodo` |
| `$order` | ordenar | `:id` (orden interno estable) |
| `$limit` / `$offset` | paginar | 50 000 filas por página |

**Por qué así.** El conjunto tiene unos 7 millones de filas (2010 a 2022) y 51 columnas, todas como texto. No necesitamos todo: pedimos **solo el periodo más reciente** y **solo las columnas que vamos a usar**. Eso reduce la descarga de varios gigabytes a unos pocos megabytes.

> **Errores comunes que evitamos:**
> 1. Descargar el conjunto completo "por si acaso".
> 2. Paginar sin `$order`: sin un orden fijo, dos páginas pueden repetir o saltarse filas.
> 3. Dejar que pandas convierta textos como `"NA"` en valores faltantes: leemos todo como texto (`dtype=str`) y solo la celda vacía cuenta como faltante.

Primero, una función para llamar a la API con reintentos (el portal a veces responde lento):

In [ ]:
def socrata(params: dict, formato: str = "json", intentos: int = 6) -> requests.Response:
    """GET a la API con reintentos (la API a veces responde lento o con 5xx)."""
    url = f"{API}.{formato}"
    for intento in range(1, intentos + 1):
        try:
            respuesta = requests.get(url, params=params, timeout=180)
            respuesta.raise_for_status()
            return respuesta
        except requests.RequestException as error:
            if intento == intentos:
                raise
            espera = 2 ** intento
            print(f"   aviso: {error}. Reintento {intento}/{intentos - 1} en {espera} s")
            time.sleep(espera)
    raise RuntimeError("inalcanzable")


def consultar_periodos() -> pd.DataFrame:
    """Periodos disponibles y número de filas de cada uno (del más reciente al más viejo)."""
    datos = socrata({"$select": "periodo, count(*) AS filas", "$group": "periodo",
                     "$order": "periodo DESC"}).json()
    tabla = pd.DataFrame(datos)
    tabla["filas"] = tabla["filas"].astype(int)
    return tabla


def etiqueta_periodo(periodo: str) -> str:
    """'20224' -> '2022-2'. El ICFES usa 1 para el primer semestre y 2, 3 o 4 para el segundo."""
    if len(periodo) == 5 and periodo.isdigit():
        return f"{periodo[:4]}-{1 if periodo[4] == '1' else 2}"
    return periodo

¿Qué periodos hay y cuántas filas tiene cada uno? Es la consulta `$select=periodo,count(*)&$group=periodo&$order=periodo DESC`. El código del periodo es el año más un dígito: `1` es la aplicación del primer semestre (calendario B) y `2`, `3` o `4` la del segundo semestre (calendario A, la más grande).

In [ ]:
periodos = consultar_periodos()
display(periodos.head(8))
if PERIODO is None:
    PERIODO = str(periodos.iloc[0]["periodo"])
print("Periodo elegido:", PERIODO, f"({etiqueta_periodo(PERIODO)})")

Ahora la descarga del periodo elegido, página por página. La primera vez tarda uno o dos minutos; después se usa la copia guardada en `datos/`.

In [ ]:
def descargar_periodo(periodo: str, columnas: list[str], carpeta: Path,
                      tam_pagina: int = 50_000) -> pd.DataFrame:
    """Descarga SOLO un periodo, por páginas, y lo guarda en caché como CSV."""
    ruta = carpeta / f"saber11_{periodo}.csv"
    if ruta.exists():
        print(f"   usando la copia local {ruta}")
        return pd.read_csv(ruta, dtype=str, keep_default_na=False, na_values=[""])

    filtro = f"periodo = '{periodo}'"
    total = int(socrata({"$select": "count(*) AS filas", "$where": filtro}).json()[0]["filas"])
    print(f"   {total:,} filas en el periodo {periodo}".replace(",", "."))
    partes = []
    for inicio in range(0, total, tam_pagina):
        respuesta = socrata({
            "$select": ",".join(columnas),
            "$where": filtro,
            "$order": ":id",          # orden estable para que las páginas no se crucen
            "$limit": tam_pagina,
            "$offset": inicio,
        }, formato="csv")
        # dtype=str: todo llega como texto. keep_default_na=False evita que pandas
        # convierta textos como "NA" en faltantes; solo la celda vacía es faltante.
        partes.append(pd.read_csv(io.StringIO(respuesta.text), dtype=str,
                                  keep_default_na=False, na_values=[""]))
        print(f"   {min(inicio + tam_pagina, total):>9,} / {total:,}".replace(",", "."))
    datos = pd.concat(partes, ignore_index=True)
    if len(datos) != total:
        raise RuntimeError(f"Se esperaban {total} filas y llegaron {len(datos)}")
    carpeta.mkdir(parents=True, exist_ok=True)
    datos.to_csv(ruta, index=False)
    return datos

In [ ]:
columnas = COLUMNAS_EXTRA + CANDIDATAS + [OBJETIVO]
crudos = descargar_periodo(PERIODO, columnas, CARPETA_DATOS)
print(f"{fmt(len(crudos))} filas y {crudos.shape[1]} columnas")
print("Tipos de dato:", crudos.dtypes.astype(str).value_counts().to_dict())
crudos.head()

Todas las columnas llegan como **texto**, incluido el puntaje. Lo convertiremos a número en la limpieza.

## 3. Exploración breve

**Qué hacemos.** Antes de modelar, miramos los datos: cuántos faltantes tiene cada variable, qué valores **reales** aparecen y cómo se relacionan con el puntaje.

**Por qué.** Las etiquetas del formulario, la limpieza y la elección de variables dependen de lo que de verdad hay en los datos, no de lo que suponemos.

> **Error común que evitamos:** escribir etiquetas o reglas de limpieza "de memoria". Un texto con o sin tilde (`BOGOTÁ` / `BOGOTA`) basta para romper un filtro o un diccionario.

Esta exploración es descriptiva: no tomamos decisiones del modelo mirando el set de prueba. Todo lo que el modelo *aprende* de los datos se ajustará solo con entrenamiento (sección 5).

In [ ]:
def resumen_columnas(datos, columnas):
    filas = []
    for columna in columnas:
        texto = datos[columna]
        conteo = texto.value_counts()
        filas.append({
            "variable": columna,
            "% faltantes": round(100 * texto.isna().mean(), 2),
            "categorías": texto.nunique(),
            "más frecuente": conteo.index[0],
            "% más frecuente": round(100 * conteo.iloc[0] / texto.notna().sum(), 1),
            "casos de la menos frecuente": int(conteo.iloc[-1]),
        })
    return pd.DataFrame(filas).set_index("variable")


resumen = resumen_columnas(crudos, CANDIDATAS)
resumen

Los valores únicos reales de cada variable (con cuántos estudiantes tiene cada uno). De aquí salen las etiquetas del formulario.

In [ ]:
for columna in CANDIDATAS:
    conteo = crudos[columna].value_counts(dropna=False)
    valores = ", ".join(f"{valor} ({fmt(n)})" for valor, n in conteo.items())
    print(f"{columna} [{len(conteo)}]: {valores}\n")

¿Cómo se distribuye el puntaje global? (Convertimos a número solo para mirar; la limpieza formal viene en la sección 4.)

In [ ]:
puntaje = pd.to_numeric(crudos[OBJETIVO], errors="coerce")
print(f"Puntajes válidos: {fmt(puntaje.notna().sum())} · faltantes: {fmt(puntaje.isna().sum())}")
print(f"Promedio {fmt(puntaje.mean(), 1)} · mediana {fmt(puntaje.median())} · desviación estándar {fmt(puntaje.std(), 1)}")
print(f"Mínimo {fmt(puntaje.min())} · máximo {fmt(puntaje.max())}")

fig, ax = plt.subplots()
ax.hist(puntaje.dropna(), bins=range(0, 505, 10), color=AZUL, edgecolor="white", linewidth=0.6)
ax.axvline(puntaje.mean(), color=TINTA, linewidth=1.5)
ax.annotate(f"promedio {fmt(puntaje.mean())}", xy=(puntaje.mean(), ax.get_ylim()[1] * 0.92),
            xytext=(8, 0), textcoords="offset points", color=TINTA, fontsize=9)
ax.set(title="Distribución del puntaje global", xlabel="Puntaje global (0 a 500)", ylabel="Estudiantes", xlim=(0, 500))
ax.yaxis.set_major_formatter(lambda y, _: fmt(y))
ax.grid(axis="x", visible=False)
plt.show()

La distribución tiene forma de campana, algo cargada a la derecha, y casi nadie llega a los extremos: la escala va de 0 a 500, pero los puntajes se concentran en un rango mucho más estrecho.

Ahora, el puntaje promedio según dos variables de contexto: el estrato y la educación de la madre. Cada barra muestra el promedio y, entre paréntesis, cuántos estudiantes hay en esa categoría.

In [ ]:
def promedio_por(columna, ax, titulo):
    tabla = (pd.DataFrame({"categoria": crudos[columna].fillna("(faltante)"), "puntaje": puntaje})
             .dropna(subset=["puntaje"]).groupby("categoria")["puntaje"].agg(["mean", "size"])
             .sort_values("mean"))
    etiquetas = [f"{c} ({fmt(n)})" for c, n in zip(tabla.index, tabla["size"])]
    ax.barh(etiquetas, tabla["mean"], color=AZUL, height=0.6)
    for y, valor in enumerate(tabla["mean"]):
        ax.text(valor + 3, y, fmt(valor), va="center", fontsize=9, color=TINTA)
    ax.axvline(puntaje.mean(), color=TINTA, linewidth=1)
    ax.set(title=titulo, xlabel="Puntaje global promedio", xlim=(0, tabla["mean"].max() * 1.15))
    ax.grid(axis="y", visible=False)
    return tabla


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
por_estrato = promedio_por("fami_estratovivienda", ax1, "Promedio por estrato")
por_madre = promedio_por("fami_educacionmadre", ax2, "Promedio por educación de la madre")
fig.text(0.01, -0.02, "La línea vertical es el promedio nacional del periodo.", color=TINTA_2)
plt.tight_layout()
plt.show()

brecha = por_estrato["mean"].max() - por_estrato["mean"].min()
print(f"Entre el estrato con mayor promedio ({por_estrato['mean'].idxmax()}) y el de menor "
      f"({por_estrato['mean'].idxmin()}) hay {fmt(brecha)} puntos de diferencia.")

El patrón es claro: a más estrato y más educación de la madre, mayor puntaje promedio. Esto es **correlación**: el estrato no "causa" el puntaje; resume condiciones del hogar (ingresos, tiempo, acceso a libros y a internet, calidad del colegio) que sí se relacionan con él.

Una mirada rápida al tipo de colegio y a la zona:

In [ ]:
tabla = (pd.DataFrame({"Tipo de colegio": crudos["cole_naturaleza"], "Zona": crudos["cole_area_ubicacion"], "puntaje": puntaje})
         .pivot_table(index="Tipo de colegio", columns="Zona", values="puntaje", aggfunc="mean"))
tabla.round(1)

## 4. Limpieza y elección de variables

**Qué hacemos.**
1. Pasamos `punt_global` a número (`errors="coerce"` convierte lo que no sea número en faltante).
2. Quitamos las filas **sin objetivo** (no hay nada que aprender de ellas) y las que estén fuera de 0 a 500.
3. Nos quedamos con los resultados publicados (`estu_estadoinvestigacion = PUBLICAR`) si el periodo tiene esa marca: los demás estaban bajo revisión.
4. En las variables categóricas, cambiamos los faltantes por la categoría **«Sin información»**.

**Por qué «Sin información» y no borrar filas ni rellenar con la moda.** Borrar filas con algún faltante bota muchos estudiantes y sesga la muestra: quienes no responden no son un grupo al azar. Rellenar con la moda inventa datos. Una categoría propia es honesta y deja que el modelo aprenda si "no responder" se asocia con el puntaje.

> **Error común que evitamos:** limpiar "a ojo" en celdas sueltas. Todo queda en una función que se aplica igual hoy, en el re-entrenamiento y en `train.py`.

La función `clave()` normaliza textos para compararlos sin tildes ni mayúsculas (así `BOGOTÁ` y `Bogota` son lo mismo).

In [ ]:
def clave(texto: str) -> str:
    """Normaliza un texto para compararlo: sin tildes, mayúsculas y espacios simples."""
    sin_tildes = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return " ".join(sin_tildes.upper().split())


def limpiar(datos: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
    datos = datos.copy()
    datos[OBJETIVO] = pd.to_numeric(datos[OBJETIVO], errors="coerce")
    datos = datos[datos[OBJETIVO].between(0, 500)]  # también descarta los NaN
    if "estu_estadoinvestigacion" in datos.columns:
        # Solo resultados publicados (quita los que estaban bajo investigación).
        estado = datos["estu_estadoinvestigacion"].fillna("").map(clave)
        if (estado == "PUBLICAR").any():
            datos = datos[estado == "PUBLICAR"]
    for columna in variables:
        texto = datos[columna].fillna("").astype(str).str.strip()
        datos[columna] = texto.replace("", SIN_INFO)
    return datos.reset_index(drop=True)

In [ ]:
if "estu_estadoinvestigacion" in crudos.columns:
    print("Estados de investigación:", crudos["estu_estadoinvestigacion"].value_counts(dropna=False).to_dict())

datos = limpiar(crudos, CANDIDATAS)
print(f"Filas descargadas: {fmt(len(crudos))}")
print(f"Filas limpias:     {fmt(len(datos))} (se quitaron {fmt(len(crudos) - len(datos))})")
promedio_nacional = float(datos[OBJETIVO].mean())
print(f"Promedio nacional del periodo (datos limpios): {fmt(promedio_nacional, 1)}")
(datos[CANDIDATAS] == SIN_INFO).sum().rename("filas con «Sin información»").to_frame()

### Qué variables usamos

El formulario de la página debe tener entre 8 y 10 campos. Partimos de las 11 candidatas y aplicamos tres criterios, mirando la tabla de la sección 3:

- **Calidad:** pocos faltantes y categorías con suficientes estudiantes.
- **Información nueva:** una variable que casi repite a otra suma poco y alarga el formulario.
- **Claridad para quien llena el formulario:** una persona debe poder responder sin buscar documentos.

(justificación pendiente de los datos reales)

> **Error común que evitamos:** elegir variables mirando el desempeño en el set de prueba. Si probamos combinaciones hasta que el test mejore, el test deja de ser una evaluación honesta.

In [ ]:
VARIABLES = ['fami_estratovivienda', 'fami_educacionmadre', 'fami_educacionpadre', 'fami_tieneinternet', 'fami_tienecomputador', 'fami_tieneautomovil', 'cole_naturaleza', 'cole_jornada', 'cole_area_ubicacion', 'cole_bilingue', 'cole_depto_ubicacion']
print(f"{len(VARIABLES)} variables en el modelo (campos del formulario):")
for variable in VARIABLES:
    print("  -", variable)
print("Fuera del modelo:", [v for v in CANDIDATAS if v not in VARIABLES])

## 5. Separación en entrenamiento y prueba

**Qué hacemos.** Separamos los datos: el 80 % para entrenar y el 20 % para evaluar. La semilla (`random_state`) hace que la separación sea la misma cada vez.

**Por qué ahora.** La separación va **antes de ajustar cualquier cosa**. Todo lo que el modelo aprende de los datos (qué categorías son raras, qué columnas crea el codificador, los coeficientes) debe salir **solo** del entrenamiento. El set de prueba simula estudiantes que el modelo nunca vio.

> **Error común que evitamos: la fuga de información (*data leakage*).** Si decidimos qué categorías son "raras" contando en todos los datos, o si hacemos `pd.get_dummies` antes de separar, el set de prueba ya influyó en el modelo y la evaluación sale optimista.

Después de separar, agrupamos las categorías con **muy pocos casos**: menos de `MIN_CASOS` estudiantes **en el entrenamiento** pasan a «Otra (pocos casos)». Con tan pocos datos, su coeficiente sería ruido. La regla aprendida en train se aplica igual al test.

In [ ]:
def aprender_frecuentes(train: pd.DataFrame, variables: list[str], min_casos: int) -> dict:
    frecuentes = {}
    for columna in variables:
        conteo = train[columna].value_counts()
        frecuentes[columna] = set(conteo[conteo >= min_casos].index)
    return frecuentes


def agrupar_raras(datos: pd.DataFrame, frecuentes: dict) -> pd.DataFrame:
    datos = datos.copy()
    for columna, conservar in frecuentes.items():
        datos[columna] = datos[columna].where(datos[columna].isin(conservar), OTRA)
    return datos

In [ ]:
x = datos[VARIABLES]
y = datos[OBJETIVO]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=PROPORCION_PRUEBA, random_state=SEMILLA)

frecuentes = aprender_frecuentes(x_train, VARIABLES, MIN_CASOS)   # se aprende SOLO con train
raras = {v: sorted(set(x_train[v]) - frecuentes[v]) for v in VARIABLES}
raras = {v: r for v, r in raras.items() if r}
x_train = agrupar_raras(x_train, frecuentes)
x_test = agrupar_raras(x_test, frecuentes)                        # misma regla para test

print(f"Entrenamiento: {fmt(len(x_train))} estudiantes · Prueba: {fmt(len(x_test))} estudiantes")
print(f"Categorías agrupadas (menos de {MIN_CASOS} casos en train):", raras or "ninguna")

## 6. Pipeline y entrenamiento

**Qué hacemos.** Armamos un `Pipeline` de scikit-learn con dos pasos:

1. **`OneHotEncoder(handle_unknown="ignore")`** convierte cada categoría en una columna de ceros y unos. Por ejemplo, `fami_estratovivienda = "Estrato 3"` se vuelve un 1 en la columna *Estrato 3* y 0 en las demás. Con `handle_unknown="ignore"`, una categoría que no apareció en el entrenamiento se codifica como todo ceros: no suma ni resta. La página hace exactamente lo mismo.
2. **`Ridge`**, una regresión lineal con una penalización pequeña (`alpha = 1`).

**Por qué Ridge y no `LinearRegression`.** Con one-hot completo, las columnas de cada variable siempre suman 1, lo mismo que el intercepto (la "trampa de las variables ficticias"): hay infinitas soluciones con el mismo error. La penalización de Ridge escoge una única solución estable. Con cientos de miles de filas, `alpha = 1` casi no cambia las predicciones.

**Por qué es tan fácil de desplegar.** Como todo es categórico, la predicción es **el intercepto más la suma de los coeficientes de las categorías elegidas**. Son unas pocas sumas: cualquier navegador las hace sin Python.

> **Error común que evitamos:** codificar a mano o con `pd.get_dummies` por separado en train y test. Las columnas pueden no coincidir y, en producción, una categoría nueva rompe todo. El pipeline guarda el codificador junto con el modelo.

In [ ]:
def crear_modelo(variables: list[str], alpha: float = ALPHA) -> Pipeline:
    preprocesamiento = ColumnTransformer(
        [("categoricas", OneHotEncoder(handle_unknown="ignore"), variables)])
    return Pipeline([("preprocesamiento", preprocesamiento),
                     ("regresion", Ridge(alpha=alpha))])

In [ ]:
modelo = crear_modelo(VARIABLES).fit(x_train, y_train)
codificador = modelo.named_steps["preprocesamiento"].named_transformers_["categoricas"]
print(f"Columnas después del one-hot: {len(codificador.get_feature_names_out())}")
print(f"Intercepto: {fmt(modelo.named_steps['regresion'].intercept_, 2)}")
modelo

## 7. Evaluación contra una línea base

**Qué hacemos.** Medimos el error en el set de prueba con tres métricas:

- **MAE** (error absoluto medio): en promedio, ¿por cuántos puntos se equivoca? Es la más fácil de explicar.
- **RMSE** (raíz del error cuadrático medio): parecida, pero castiga más los errores grandes.
- **R²**: qué parte de la variación del puntaje explica el modelo. 0 equivale a responder siempre el promedio; 1 sería perfecto.

Y lo comparamos con una **línea base**: un `DummyRegressor` que siempre responde el promedio del entrenamiento.

**Por qué una línea base.** Un MAE de 35 puntos no dice nada solo. ¿Es bueno? Depende de cuánto se equivoca "no saber nada". El modelo vale la pena en la medida en que le gana a esa referencia.

**Comparación opcional.** Entrenamos también un `HistGradientBoostingRegressor`, un modelo más flexible que captura interacciones. Si le gana por mucho al lineal, perdemos precisión por quedarnos con un modelo interpretable; si le gana por poco, el lineal es la mejor opción para explicar y desplegar.

> **Error común que evitamos:** reportar métricas sin línea base, o medir en los mismos datos con los que se entrenó.

In [ ]:
def metricas(y_real, y_pred) -> dict:
    return {
        "mae": float(mean_absolute_error(y_real, y_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_real, y_pred))),
        "r2": float(r2_score(y_real, y_pred)),
    }


def crear_boosting(variables: list[str]) -> Pipeline:
    codificador = ColumnTransformer([("categoricas", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1), variables)])
    boosting = HistGradientBoostingRegressor(
        categorical_features=list(range(len(variables))), random_state=SEMILLA)
    return Pipeline([("preprocesamiento", codificador), ("regresion", boosting)])

In [ ]:
base = DummyRegressor(strategy="mean").fit(x_train, y_train)
boosting = crear_boosting(VARIABLES).fit(x_train, y_train)

pred_modelo = modelo.predict(x_test)
resultados = {
    "modelo": {"nombre": "Regresión lineal (Ridge)", **metricas(y_test, pred_modelo)},
    "linea_base": {"nombre": "Línea base: siempre el promedio", **metricas(y_test, base.predict(x_test))},
    "boosting": {"nombre": "HistGradientBoosting (solo comparación)", **metricas(y_test, boosting.predict(x_test))},
}
tabla_metricas = pd.DataFrame(resultados.values()).set_index("nombre")
display(tabla_metricas.round(3))

mae = resultados["modelo"]["mae"]
mejora = 1 - mae / resultados["linea_base"]["mae"]
cobertura = float(np.mean(np.abs(y_test.to_numpy() - pred_modelo) <= mae))
print(f"El modelo lineal se equivoca en promedio {fmt(mae, 1)} puntos; la línea base, "
      f"{fmt(resultados['linea_base']['mae'], 1)}. Mejora del {fmt(100 * mejora)} %.")
print(f"R² = {fmt(resultados['modelo']['r2'], 3)}: el contexto explica cerca del "
      f"{fmt(100 * resultados['modelo']['r2'])} % de la variación del puntaje.")
print(f"Gradient boosting: MAE {fmt(resultados['boosting']['mae'], 1)} "
      f"({fmt(resultados['linea_base']['mae'] - resultados['boosting']['mae'], 1)} puntos mejor que la base; "
      f"el lineal, {fmt(resultados['linea_base']['mae'] - mae, 1)}).")
print(f"En el {fmt(100 * cobertura)} % de los estudiantes de prueba, el puntaje real quedó dentro de ± MAE.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.1]})
nombres = ["Línea base", "Ridge (desplegado)", "Gradient boosting"]
valores = [resultados["linea_base"]["mae"], mae, resultados["boosting"]["mae"]]
ax1.barh(nombres, valores, color=[GRIS, AZUL, GRIS], height=0.55)
for y_pos, valor in enumerate(valores):
    ax1.text(valor + 0.4, y_pos, fmt(valor, 1), va="center", color=TINTA)
ax1.invert_yaxis()
ax1.set(title="Error absoluto medio en el set de prueba", xlabel="MAE (puntos; menos es mejor)", xlim=(0, max(valores) * 1.18))
ax1.grid(axis="y", visible=False)

hb = ax2.hexbin(pred_modelo, y_test, gridsize=45, cmap="Blues", mincnt=1, bins="log", linewidths=0)
ax2.plot([0, 500], [0, 500], color=TINTA, linewidth=1)
ax2.set(title="Cada estudiante de prueba: estimado vs. real", xlabel="Puntaje estimado", ylabel="Puntaje real",
        xlim=(100, 450), ylim=(0, 500))
fig.colorbar(hb, ax=ax2, label="Estudiantes (escala log)")
plt.tight_layout()
plt.show()

**Cómo leer la gráfica de la derecha.** Si el modelo fuera perfecto, todos los puntos estarían sobre la diagonal. En cambio, para un mismo puntaje estimado hay estudiantes reales muy por encima y muy por debajo. Esa es la razón técnica detrás del aviso ético: el modelo describe **promedios de grupos**, no personas.

## 8. Interpretación de los coeficientes

**Qué hacemos.** Leemos los coeficientes del modelo lineal. Dentro de una misma variable, la **diferencia** entre los coeficientes de dos categorías dice cuánto cambia la estimación al pasar de una a otra, **dejando lo demás igual**.

Para que sea fácil de leer, comparamos cada categoría con la **más frecuente** de su variable (la referencia): *«pasar de X a Y suma o resta N puntos, dejando lo demás igual»*.

> **Errores comunes que evitamos:**
> 1. Leer un coeficiente solo, sin comparar. Con one-hot completo, el nivel de cada coeficiente es arbitrario; lo que tiene sentido es la diferencia entre categorías de la misma variable.
> 2. Leerlos como causas. "Pasar de no tener internet a tenerlo suma N puntos" describe a estudiantes distintos, no lo que pasaría si a alguien le instalan internet.

In [ ]:
coeficientes = modelo.named_steps["regresion"].coef_
filas, inicio = [], 0
for i, variable in enumerate(VARIABLES):
    categorias = codificador.categories_[i]
    coefs = coeficientes[inicio:inicio + len(categorias)]
    inicio += len(categorias)
    conteo = x_train[variable].value_counts()
    referencia = conteo.idxmax()
    coef_ref = coefs[list(categorias).index(referencia)]
    for categoria, coef in zip(categorias, coefs):
        filas.append({"variable": variable, "categoría": categoria, "coeficiente": coef,
                      "vs. más frecuente": coef - coef_ref, "referencia": referencia,
                      "estudiantes (train)": int(conteo.get(categoria, 0))})
tabla_coef = pd.DataFrame(filas)
tabla_coef.round(2)

La misma información en frases: para cada variable, qué pasa al cambiar la respuesta más común por la categoría con mayor y con menor coeficiente.

In [ ]:
def frase(puntos):
    return f"suma {fmt(puntos, 1)}" if puntos >= 0 else f"resta {fmt(-puntos, 1)}"


for variable, grupo in tabla_coef.groupby("variable", sort=False):
    ref = grupo["referencia"].iloc[0]
    alta = grupo.loc[grupo["vs. más frecuente"].idxmax()]
    baja = grupo.loc[grupo["vs. más frecuente"].idxmin()]
    print(f"{variable}: pasar de «{ref}» a «{alta['categoría']}» {frase(alta['vs. más frecuente'])} puntos; "
          f"a «{baja['categoría']}» {frase(baja['vs. más frecuente'])} puntos.")

In [ ]:
otras = [v for v in VARIABLES if v != "cole_depto_ubicacion"]
columnas_grafica = 2
filas_grafica = math.ceil(len(otras) / columnas_grafica)
alturas = [max(len(tabla_coef[tabla_coef.variable == v]) for v in otras[i:i + columnas_grafica]) + 1.5
           for i in range(0, len(otras), columnas_grafica)]
fig, ejes = plt.subplots(filas_grafica, columnas_grafica, figsize=(13, sum(alturas) * 0.32 + 1),
                         gridspec_kw={"height_ratios": alturas}, sharex=True)
limite = tabla_coef["vs. más frecuente"].abs().max() * 1.15
for ax, variable in zip(ejes.flat, otras):
    grupo = tabla_coef[tabla_coef.variable == variable].sort_values("vs. más frecuente")
    colores = [GRIS if abs(d) < 1e-9 else (AZUL if d > 0 else ROJO) for d in grupo["vs. más frecuente"]]
    ax.barh(grupo["categoría"], grupo["vs. más frecuente"], color=colores, height=0.6)
    ax.axvline(0, color=TINTA_2, linewidth=1)
    ax.set_title(f"{variable} (referencia: {grupo['referencia'].iloc[0]})", fontsize=10)
    ax.set_xlim(-limite, limite)
    ax.grid(axis="y", visible=False)
for ax in list(ejes.flat)[len(otras):]:
    ax.set_visible(False)
fig.suptitle("Puntos que suma (azul) o resta (rojo) cada categoría frente a la más frecuente", x=0.01, ha="left",
             fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
grupo = tabla_coef[tabla_coef.variable == "cole_depto_ubicacion"].sort_values("vs. más frecuente")
fig, ax = plt.subplots(figsize=(9, len(grupo) * 0.26 + 1))
colores = [GRIS if abs(d) < 1e-9 else (AZUL if d > 0 else ROJO) for d in grupo["vs. más frecuente"]]
ax.barh(grupo["categoría"], grupo["vs. más frecuente"], color=colores, height=0.65)
ax.axvline(0, color=TINTA_2, linewidth=1)
ax.set(title=f"Departamento del colegio frente a {grupo['referencia'].iloc[0]}, dejando lo demás igual",
       xlabel="Puntos frente a la referencia")
ax.grid(axis="y", visible=False)
plt.show()

**Para discutir.** Los coeficientes de una variable están "ajustados" por las demás. Por ejemplo, el efecto del estrato aquí es menor que la brecha cruda de la sección 3, porque parte de esa brecha ya la explican la educación de los padres, el tipo de colegio o el acceso a internet. Esto es lo que significa *dejando lo demás igual*.

## 9. Exportación a `modelo.js` y prueba de paridad

**Qué hacemos.** Guardamos el modelo como **números**: el intercepto y, por cada variable, su lista de categorías con su coeficiente y una etiqueta legible en español. Lo escribimos en dos archivos:

- `web/modelo.js` define `window.MODELO = {...}`. Funciona tanto al abrir `index.html` con doble clic como en Netlify (un `fetch` de un `.json` fallaría al abrir el archivo directamente).
- `web/modelo.json`, el mismo contenido para quien quiera leerlo o usarlo desde otro programa.

**Por qué números y no un `.pkl`.** Netlify publica páginas estáticas y sus funciones no ejecutan Python (solo JavaScript/TypeScript, y Go mediante la API compatible con Lambda). Un `.pkl` de scikit-learn no podría correr allí. Pero nuestro modelo es una suma: basta con los números y diez líneas de JavaScript.

Primero, las etiquetas legibles y el orden de las categorías para el formulario. Se escribieron a partir de los valores únicos reales de la sección 3; si un valor no tiene etiqueta, la exportación avisa.

In [ ]:
# Nombre, etiqueta corta, grupo y ayuda de cada variable (para el formulario).
INFO_VARIABLES = {
    "fami_estratovivienda": ("Estrato de la vivienda", "Estrato", "hogar",
                             "Aparece en los recibos de servicios públicos."),
    "fami_educacionmadre": ("Nivel educativo de la madre", "Educación de la madre", "hogar",
                            "El nivel más alto que alcanzó."),
    "fami_educacionpadre": ("Nivel educativo del padre", "Educación del padre", "hogar",
                            "El nivel más alto que alcanzó."),
    "fami_tieneinternet": ("¿Hay internet en el hogar?", "Internet en el hogar", "hogar", ""),
    "fami_tienecomputador": ("¿Hay computador en el hogar?", "Computador en el hogar", "hogar", ""),
    "fami_tieneautomovil": ("¿La familia tiene automóvil?", "Automóvil", "hogar", ""),
    "cole_naturaleza": ("Tipo de colegio", "Tipo de colegio", "colegio",
                        "Oficial es público; no oficial es privado."),
    "cole_jornada": ("Jornada del colegio", "Jornada", "colegio", ""),
    "cole_area_ubicacion": ("Zona donde está el colegio", "Zona del colegio", "colegio", ""),
    "cole_bilingue": ("¿El colegio es bilingüe?", "Colegio bilingüe", "colegio", ""),
    "cole_depto_ubicacion": ("Departamento del colegio", "Departamento", "colegio", ""),
}

# Etiquetas legibles y orden lógico de las categorías. Las llaves se comparan
# sin tildes y en mayúsculas (ver clave()), así que toleran variantes como
# "BOGOTÁ"/"BOGOTA". Revisadas contra los valores únicos reales del periodo.
EDUCACION = [
    ("NINGUNO", "Ninguno"),
    ("PRIMARIA INCOMPLETA", "Primaria incompleta"),
    ("PRIMARIA COMPLETA", "Primaria completa"),
    ("SECUNDARIA (BACHILLERATO) INCOMPLETA", "Bachillerato incompleto"),
    ("SECUNDARIA (BACHILLERATO) COMPLETA", "Bachillerato completo"),
    ("TECNICA O TECNOLOGICA INCOMPLETA", "Técnica o tecnológica incompleta"),
    ("TECNICA O TECNOLOGICA COMPLETA", "Técnica o tecnológica completa"),
    ("EDUCACION PROFESIONAL INCOMPLETA", "Universitaria incompleta"),
    ("EDUCACION PROFESIONAL COMPLETA", "Universitaria completa"),
    ("POSTGRADO", "Posgrado"),
    ("NO SABE", "No sabe"),
    ("NO APLICA", "No aplica"),
]
SI_NO = [("SI", "Sí"), ("S", "Sí"), ("NO", "No"), ("N", "No")]
CATEGORIAS = {
    "fami_estratovivienda": [
        ("ESTRATO 1", "Estrato 1"), ("ESTRATO 2", "Estrato 2"), ("ESTRATO 3", "Estrato 3"),
        ("ESTRATO 4", "Estrato 4"), ("ESTRATO 5", "Estrato 5"), ("ESTRATO 6", "Estrato 6"),
        ("SIN ESTRATO", "Sin estrato"),
    ],
    "fami_educacionmadre": EDUCACION,
    "fami_educacionpadre": EDUCACION,
    "fami_tieneinternet": SI_NO,
    "fami_tienecomputador": SI_NO,
    "fami_tieneautomovil": SI_NO,
    "cole_naturaleza": [("OFICIAL", "Oficial (público)"), ("NO OFICIAL", "No oficial (privado)")],
    "cole_jornada": [
        ("MANANA", "Mañana"), ("TARDE", "Tarde"), ("COMPLETA", "Completa"), ("UNICA", "Única"),
        ("NOCHE", "Noche"), ("SABATINA", "Sabatina"),
    ],
    "cole_area_ubicacion": [("URBANO", "Urbana"), ("RURAL", "Rural")],
    "cole_bilingue": SI_NO,
    "cole_depto_ubicacion": [
        ("AMAZONAS", "Amazonas"), ("ANTIOQUIA", "Antioquia"), ("ARAUCA", "Arauca"),
        ("ATLANTICO", "Atlántico"), ("BOGOTA", "Bogotá D. C."), ("BOGOTA D.C.", "Bogotá D. C."),
        ("BOGOTA, D.C.", "Bogotá D. C."), ("BOLIVAR", "Bolívar"), ("BOYACA", "Boyacá"),
        ("CALDAS", "Caldas"), ("CAQUETA", "Caquetá"), ("CASANARE", "Casanare"),
        ("CAUCA", "Cauca"), ("CESAR", "Cesar"), ("CHOCO", "Chocó"), ("CORDOBA", "Córdoba"),
        ("CUNDINAMARCA", "Cundinamarca"), ("GUAINIA", "Guainía"), ("GUAVIARE", "Guaviare"),
        ("HUILA", "Huila"), ("LA GUAJIRA", "La Guajira"), ("GUAJIRA", "La Guajira"),
        ("MAGDALENA", "Magdalena"), ("META", "Meta"), ("NARINO", "Nariño"),
        ("NORTE SANTANDER", "Norte de Santander"), ("NORTE DE SANTANDER", "Norte de Santander"),
        ("PUTUMAYO", "Putumayo"), ("QUINDIO", "Quindío"), ("RISARALDA", "Risaralda"),
        ("SAN ANDRES", "San Andrés y Providencia"),
        ("ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA", "San Andrés y Providencia"),
        ("SANTANDER", "Santander"), ("SUCRE", "Sucre"), ("TOLIMA", "Tolima"),
        ("VALLE", "Valle del Cauca"), ("VALLE DEL CAUCA", "Valle del Cauca"),
        ("VAUPES", "Vaupés"), ("VICHADA", "Vichada"), ("EXTRANJERO", "Extranjero"),
    ],
}
# Variables sin orden natural: sus categorías se ordenan alfabéticamente por etiqueta.
ORDEN_ALFABETICO = {"cole_depto_ubicacion"}

La función de exportación. Además del modelo guarda la ficha técnica (periodo, tamaños, métricas) y la **frecuencia** de cada categoría en train. Con las frecuencias se cumple una identidad útil para la página: *intercepto + Σ (frecuencia × coeficiente) = promedio de entrenamiento*. Así la página puede mostrar cuánto aporta cada respuesta frente al promedio.

In [ ]:
def etiqueta_categoria(variable: str, valor: str) -> str:
    if valor == SIN_INFO:
        return SIN_INFO
    if valor == OTRA:
        return OTRA
    for llave, etiqueta in CATEGORIAS.get(variable, []):
        if clave(valor) == llave:
            return etiqueta
    print(f"   aviso: sin etiqueta para {variable} = {valor!r}; se usa el texto original")
    return valor.capitalize()


def posicion_categoria(variable: str, valor: str, etiqueta: str) -> tuple:
    """Llave de orden: orden lógico del diccionario; al final 'Sin información' y 'Otra'."""
    if valor == SIN_INFO:
        return (2, 0, "")
    if valor == OTRA:
        return (3, 0, "")
    if variable in ORDEN_ALFABETICO:
        return (0, 0, clave(etiqueta))
    llaves = [llave for llave, _ in CATEGORIAS.get(variable, [])]
    return (0, llaves.index(clave(valor)), "") if clave(valor) in llaves else (1, 0, clave(etiqueta))


def exportar_modelo(modelo: Pipeline, variables: list[str], x_train: pd.DataFrame,
                    y_train: pd.Series, extras: dict) -> dict:
    """Arma el diccionario que usa la página: intercepto + coeficientes por categoría."""
    codificador = modelo.named_steps["preprocesamiento"].named_transformers_["categoricas"]
    ridge = modelo.named_steps["regresion"]
    salida_variables = []
    inicio = 0
    for indice, variable in enumerate(variables):
        categorias = list(codificador.categories_[indice])
        coeficientes = ridge.coef_[inicio:inicio + len(categorias)]
        inicio += len(categorias)
        conteo = x_train[variable].value_counts()
        filas = []
        for valor, coeficiente in zip(categorias, coeficientes):
            etiqueta = etiqueta_categoria(variable, valor)
            fila = {
                "valor": str(valor),
                "etiqueta": etiqueta,
                "coeficiente": float(coeficiente),
                "frecuencia": float(conteo.get(valor, 0) / len(x_train)),
                "n": int(conteo.get(valor, 0)),
            }
            if valor == OTRA:
                agrupadas = [etiqueta_categoria(variable, r) for r in extras["raras"].get(variable, [])]
                fila["agrupa"] = sorted(set(agrupadas), key=clave)
            filas.append(fila)
        filas.sort(key=lambda f: posicion_categoria(variable, f["valor"], f["etiqueta"]))
        nombre, corta, grupo, ayuda = INFO_VARIABLES[variable]
        salida_variables.append({
            "nombre": variable,
            "etiqueta": nombre,
            "etiqueta_corta": corta,
            "grupo": grupo,
            "ayuda": ayuda,
            "mas_frecuente": str(conteo.idxmax()),
            "categorias": filas,
        })
    if inicio != len(ridge.coef_):
        raise RuntimeError("El número de coeficientes no coincide con las categorías")

    # Identidad útil para la gráfica de contribuciones:
    # intercepto + sum(frecuencia * coeficiente) = promedio de y en train.
    promedio_train = float(np.mean(y_train))
    reconstruido = float(ridge.intercept_) + sum(
        c["frecuencia"] * c["coeficiente"] for v in salida_variables for c in v["categorias"])
    if abs(reconstruido - promedio_train) > 1e-6:
        raise RuntimeError(f"La identidad del promedio falla: {reconstruido} vs {promedio_train}")

    return {
        "version": VERSION,
        "proyecto": "Predictor Saber 11",
        "fuente": {
            "nombre": "Resultados únicos Saber 11 (ICFES)",
            "portal": "datos.gov.co",
            "id": DATASET_ID,
            "url": URL_CONJUNTO,
        },
        "periodo": extras["periodo"],
        "periodo_etiqueta": extras["periodo_etiqueta"],
        "fecha_entrenamiento": extras["fecha"],
        "n_total": extras["n_total"],
        "n_entrenamiento": int(len(x_train)),
        "n_prueba": extras["n_prueba"],
        "semilla": SEMILLA,
        "algoritmo": f"Ridge (alpha = {ALPHA:g}) con codificación one-hot",
        "sklearn_version": sklearn.__version__,
        "objetivo": {"nombre": OBJETIVO, "etiqueta": "Puntaje global", "minimo": 0, "maximo": 500},
        "metricas": extras["metricas"],
        "cobertura_mae": extras["cobertura_mae"],
        "promedio_nacional": extras["promedio_nacional"],
        "promedio_entrenamiento": promedio_train,
        "intercepto": float(ridge.intercept_),
        "variables": salida_variables,
        "excluidas": [
            {"nombre": "estu_genero",
             "motivo": "Decisión ética: el modelo no debe cambiar su predicción por el género."},
            {"nombre": "cole_nombre_establecimiento, cole_cod_dane_*, cole_codigo_icfes",
             "motivo": "Nombres y códigos de colegio: identifican instituciones y no describen el contexto."},
        ],
        "casos_prueba": extras["casos_prueba"],
    }


def escribir_modelo(modelo_dict: dict, carpeta_web: Path) -> tuple[Path, Path]:
    carpeta_web.mkdir(parents=True, exist_ok=True)
    texto = json.dumps(modelo_dict, ensure_ascii=False, indent=2)
    ruta_json = carpeta_web / "modelo.json"
    ruta_js = carpeta_web / "modelo.js"
    ruta_json.write_text(texto + "\n", encoding="utf-8")
    ruta_js.write_text(
        "// Generado automáticamente por scripts/train.py o por el cuaderno.\n"
        "// No lo edites a mano: vuelve a entrenar y reemplaza este archivo.\n"
        f"window.MODELO = {texto};\n", encoding="utf-8")
    return ruta_js, ruta_json


def elegir_casos(modelo: Pipeline, x_test: pd.DataFrame, y_test: pd.Series,
                 variables: list[str], cantidad: int = 5) -> list[dict]:
    """Casos del set de prueba repartidos entre predicciones bajas, medias y altas."""
    predicciones = modelo.predict(x_test)
    orden = np.argsort(predicciones, kind="stable")
    cuantiles = np.linspace(0.05, 0.95, cantidad)
    casos = []
    for numero, q in enumerate(cuantiles, start=1):
        i = int(orden[int(round(q * (len(orden) - 1)))])
        fila = x_test.iloc[[i]]
        casos.append({
            "id": numero,
            "entradas": {v: str(fila.iloc[0][v]) for v in variables},
            "puntaje_real": float(y_test.iloc[i]),
            "prediccion_sklearn": float(modelo.predict(fila)[0]),
        })
    return casos

In [ ]:
extras = {
    "periodo": PERIODO,
    "periodo_etiqueta": etiqueta_periodo(PERIODO),
    "fecha": dt.date.today().isoformat(),
    "n_total": int(len(datos)),
    "n_prueba": int(len(x_test)),
    "metricas": resultados,
    "cobertura_mae": cobertura,
    "promedio_nacional": promedio_nacional,
    "raras": raras,
    "casos_prueba": elegir_casos(modelo, x_test, y_test, VARIABLES),  # 5 estudiantes del set de prueba
}
modelo_exportado = exportar_modelo(modelo, VARIABLES, x_train, y_train, extras)
ruta_modelo_js, ruta_modelo_json = escribir_modelo(modelo_exportado, CARPETA_WEB)
print(f"{ruta_modelo_js.relative_to(RAIZ)}: {fmt(ruta_modelo_js.stat().st_size / 1024, 1)} KB")
print(f"{ruta_modelo_json.relative_to(RAIZ)}: {fmt(ruta_modelo_json.stat().st_size / 1024, 1)} KB")
print()
print("\n".join(ruta_modelo_js.read_text(encoding="utf-8").splitlines()[:24]))
print("…")

Los 5 casos de prueba que viajan dentro de `modelo.js` (repartidos entre estimaciones bajas, medias y altas):

In [ ]:
pd.DataFrame([{**caso["entradas"], "puntaje real": caso["puntaje_real"],
               "predicción sklearn": round(caso["prediccion_sklearn"], 6)} for caso in modelo_exportado["casos_prueba"]])

### Prueba de paridad: ¿JavaScript da lo mismo que scikit-learn?

**Qué hacemos.** Con **Node.js** cargamos `modelo.js` igual que lo haría el navegador, recalculamos las predicciones en JavaScript y las comparamos con las de scikit-learn. Deben coincidir con una tolerancia de **1e-6** (una millonésima de punto). Probamos los 5 casos de la página y 2 000 estudiantes más del set de prueba.

**Por qué.** Es la prueba de que lo que se despliega es exactamente lo que evaluamos. Si la página ordenara mal las categorías o confundiera una etiqueta, el error sería silencioso: la página mostraría números creíbles pero equivocados.

> **Error común que evitamos:** probar "a ojo" con un par de casos. Una prueba automática con tolerancia explícita detecta errores pequeños y se repite en cada re-entrenamiento.

El programa de JavaScript es el mismo archivo `scripts/prueba_paridad.js` del repositorio:

In [ ]:
PRUEBA_PARIDAD_JS = r"""#!/usr/bin/env node
/*
 * Prueba de paridad: la predicción en JavaScript debe coincidir con scikit-learn.
 *
 * Uso:  node scripts/prueba_paridad.js [web/modelo.js] [casos_extra.json]
 *
 * 1. Carga modelo.js igual que el navegador (el archivo define window.MODELO).
 * 2. Recalcula cada caso con la misma fórmula de web/index.html:
 *    intercepto + suma de los coeficientes de las categorías elegidas.
 * 3. Compara con la predicción de scikit-learn guardada al exportar.
 *
 * Termina con código 1 si alguna diferencia supera la tolerancia (1e-6).
 */
'use strict';

var fs = require('fs');
var path = require('path');
var vm = require('vm');

var TOLERANCIA = 1e-6;
var rutaModelo = process.argv[2] || path.join(__dirname, '..', 'web', 'modelo.js');
var rutaExtra = process.argv[3];

function cargarModelo(ruta) {
  var contexto = { window: {} };
  vm.createContext(contexto);
  vm.runInContext(fs.readFileSync(ruta, 'utf8'), contexto, { filename: ruta });
  if (!contexto.window.MODELO) {
    throw new Error(ruta + ' no definió window.MODELO');
  }
  return contexto.window.MODELO;
}

// Misma función que usa web/index.html.
function predecir(modelo, respuestas) {
  var total = modelo.intercepto;
  modelo.variables.forEach(function (variable) {
    var valor = respuestas[variable.nombre];
    var categoria = variable.categorias.find(function (c) { return c.valor === valor; });
    if (categoria) total += categoria.coeficiente; // categoría desconocida: aporta 0
  });
  return total;
}

function comparar(modelo, casos) {
  var resumen = { n: casos.length, maxDif: 0, fallos: 0, desconocidas: 0 };
  casos.forEach(function (caso) {
    modelo.variables.forEach(function (variable) {
      var valor = caso.entradas[variable.nombre];
      var existe = variable.categorias.some(function (c) { return c.valor === valor; });
      if (!existe) resumen.desconocidas += 1;
    });
    var dif = Math.abs(predecir(modelo, caso.entradas) - caso.prediccion_sklearn);
    if (dif > resumen.maxDif) resumen.maxDif = dif;
    if (!(dif <= TOLERANCIA)) resumen.fallos += 1;
  });
  return resumen;
}

function formato(numero) {
  return numero.toFixed(6).padStart(12);
}

var modelo = cargarModelo(rutaModelo);
var ok = true;

console.log('Modelo ' + modelo.version + ' · periodo ' + modelo.periodo +
            ' · ' + modelo.variables.length + ' variables');
console.log('Caso      sklearn   JavaScript    diferencia');
modelo.casos_prueba.forEach(function (caso) {
  var js = predecir(modelo, caso.entradas);
  var dif = Math.abs(js - caso.prediccion_sklearn);
  console.log(String(caso.id).padStart(4) + formato(caso.prediccion_sklearn) + ' ' +
              formato(js) + '  ' + dif.toExponential(2));
});

var casos = comparar(modelo, modelo.casos_prueba);
console.log('Casos de modelo.js: ' + casos.n + ', diferencia máxima ' +
            casos.maxDif.toExponential(2) + ', fuera de tolerancia: ' + casos.fallos);
if (casos.fallos > 0 || casos.desconocidas > 0) ok = false;

if (rutaExtra) {
  var extra = comparar(modelo, JSON.parse(fs.readFileSync(rutaExtra, 'utf8')));
  console.log('Casos extra del set de prueba: ' + extra.n + ', diferencia máxima ' +
              extra.maxDif.toExponential(2) + ', fuera de tolerancia: ' + extra.fallos);
  if (extra.fallos > 0) ok = false;
}

// La gráfica de contribuciones usa: intercepto + suma(frecuencia x coeficiente)
// = promedio del entrenamiento. Si esto falla, la gráfica no cuadraría.
var base = modelo.intercepto;
modelo.variables.forEach(function (variable) {
  variable.categorias.forEach(function (c) { base += c.frecuencia * c.coeficiente; });
});
var difBase = Math.abs(base - modelo.promedio_entrenamiento);
console.log('Identidad del promedio: diferencia ' + difBase.toExponential(2));
if (!(difBase <= TOLERANCIA)) ok = false;

console.log(ok ? 'OK: JavaScript y scikit-learn coinciden (tolerancia 1e-6).'
               : 'ERROR: la paridad falló.');
process.exit(ok ? 0 : 1);
"""

ruta_prueba = CARPETA_DATOS / "prueba_paridad.js"
ruta_prueba.write_text(PRUEBA_PARIDAD_JS, encoding="utf-8")
print("Prueba escrita en", ruta_prueba.relative_to(RAIZ))

In [ ]:
def buscar_node():
    node = shutil.which("node")
    if node is None and EN_COLAB:
        print("Instalando Node.js en Colab (unos segundos)…")
        subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y nodejs > /dev/null", shell=True)
        node = shutil.which("node")
    return node


# 2 000 estudiantes más del set de prueba, con la predicción de scikit-learn.
muestra = x_test.sample(n=min(2000, len(x_test)), random_state=SEMILLA)
casos_extra = [{"entradas": {v: str(fila[v]) for v in VARIABLES}, "prediccion_sklearn": float(p)}
               for (_, fila), p in zip(muestra.iterrows(), modelo.predict(muestra))]
ruta_extra = CARPETA_DATOS / "paridad_casos.json"
ruta_extra.write_text(json.dumps(casos_extra, ensure_ascii=False), encoding="utf-8")

node = buscar_node()
if node:
    salida = subprocess.run([node, str(ruta_prueba), str(ruta_modelo_js), str(ruta_extra)], capture_output=True, text=True)
    print(salida.stdout + salida.stderr)
    assert salida.returncode == 0, "La paridad falló: revisa la exportación antes de publicar."
else:
    # Respaldo si no hay Node: la misma cuenta en Python con los números exportados.
    print("No hay Node.js en este entorno. Verificación equivalente en Python (corre la de Node en tu computador):")
    for caso in modelo_exportado["casos_prueba"] + casos_extra:
        total = modelo_exportado["intercepto"]
        for v in modelo_exportado["variables"]:
            total += next((c["coeficiente"] for c in v["categorias"] if c["valor"] == caso["entradas"][v["nombre"]]), 0.0)
        assert abs(total - caso["prediccion_sklearn"]) <= 1e-6
    print("OK: los números exportados reproducen a scikit-learn (tolerancia 1e-6).")

## 10. Publicación y re-entrenamiento

### Cómo se publica (una sola vez)

El proyecto vive en GitHub y Netlify se conecta al repositorio: **cada push a `main` publica la página sola**.

```
Cuaderno (Colab) ──► web/modelo.js ──► GitHub (dev → main) ──► Netlify ──► https://…netlify.app
```

1. En [app.netlify.com](https://app.netlify.com): **Add new project → Import an existing project → GitHub**, y autoriza a Netlify.
2. Elige el repositorio `Ejercicio_ejemplo_machine_learning`.
3. Configura: **Branch to deploy:** `main` · **Build command:** vacío · **Publish directory:** `web` (ya viene en `netlify.toml`).
4. **Deploy**. En uno o dos minutos la página queda en una dirección `https://<nombre>.netlify.app` (el nombre se cambia en *Project configuration → Change project name*).

**Créditos del plan gratuito.** Son 300 créditos al mes con límite duro y cada publicación en producción cuesta 15: alcanzan para unas **20 publicaciones al mes**. Por eso trabajamos en la rama `dev` y fusionamos a `main` solo cuando todo está verificado. Además, `netlify.toml` omite la publicación cuando un push no cambia nada en `web/` (por ejemplo, si solo cambia este cuaderno).

**Plan B:** como el sitio es estático, GitHub Pages también sirve gratis (instrucciones en el README).

### Cómo re-entrenar

1. Abre este cuaderno en Colab y ejecuta todo (*Entorno de ejecución → Ejecutar todas*). Si quieres otro periodo, cambia `PERIODO` en la configuración.
2. Al final se descargan `modelo.js` y `modelo.json`. Súbelos a la carpeta `web/` de la rama `dev` para reemplazar los anteriores (en GitHub: entra a `web`, *Add file → Upload files*).
3. Abre `web/index.html` (o la vista previa) y revisa que la sección *Verificación* diga «5 de 5 coinciden».
4. Fusiona `dev` en `main` con un pull request. Netlify publica la nueva versión automáticamente.

Desde la terminal, `python scripts/train.py` hace lo mismo que este cuaderno y además corre la prueba de paridad.

### Para discutir en clase

1. El modelo explica una parte de la variación del puntaje (mira el R²). ¿Qué hay en la parte que no explica?
2. Si agregáramos `estu_genero`, la métrica podría mejorar un poco. ¿Por qué igual no lo hacemos? ¿Qué mensaje recibiría quien usa la página?
3. El gradient boosting casi no mejora al modelo lineal. ¿Qué nos dice eso sobre las interacciones entre variables de contexto?
4. ¿Qué pasaría si un colegio usara esta página para decidir a quién admitir? ¿Quién saldría perjudicado?
5. Los datos son de un periodo pasado. ¿Qué cambios en el país o en el examen podrían volver obsoleto el modelo?

Por último, en Colab se descargan `modelo.js` (el que usa la página) y `modelo.json` (la misma información, para consultarla) para subirlos a `web/` en el repositorio. Si el navegador pregunta si permites descargar varios archivos, acepta.

In [ ]:
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_modelo_js))
    files.download(str(ruta_modelo_json))
else:
    print(f"Fuera de Colab no hay descarga automática: los archivos quedaron en {CARPETA_WEB.relative_to(RAIZ)}/")